# 台北美食機器人：微調小模型解析條件（Colab 版）

**使用方式**
1. 上方選單「執行階段 → 變更執行階段類型」→ 選 **T4 GPU** → 儲存。
2. 選單「執行階段 → 全部執行」。
3. 中途會跳出 Google Drive 授權視窗：選同一個帳號，權限頁按「全選」再繼續（結果會存在雲端硬碟的 `taipei-food-bot-finetune` 資料夾）。
4. 全部跑完（約 2.5～3 小時）後，最後一格會下載 `results.zip`，把它交給 Claude。

**斷線或額度用完**：明天再「全部執行」一次就好，做完的步驟會自動跳過，訓練會從最後一個存檔點接著跑。

## 1. 檢查 GPU、連接 Google Drive

In [ ]:
import os, subprocess, torch
assert torch.cuda.is_available(), "沒有 GPU：請到「執行階段 → 變更執行階段類型」選 T4 GPU"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.0f} GB")

from google.colab import drive
try:
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/taipei-food-bot-finetune"
except Exception as e:
    # 授權沒完成（例如權限頁沒勾雲端硬碟）：先存在 Colab 本機，照樣能跑，但斷線後結果會不見
    print(f"⚠️ 雲端硬碟沒有連上（{e}）")
    print("   想要斷線後能接著跑：重新執行這一格，授權時權限頁按「全選」。現在先存在 Colab 本機。")
    OUT = "/content/outputs"
os.makedirs(OUT, exist_ok=True)
os.environ["FINETUNE_OUT"] = OUT
REPO = "/content/taipei-food-bot"
print("結果會存在：", OUT)

## 2. 下載程式和資料、安裝套件

In [ ]:
if not os.path.isdir(REPO):
    !git clone -q --branch finetune-parser https://github.com/god040709/taipei-food-bot.git {REPO}
else:
    !cd {REPO} && git pull -q
!cd {REPO} && git log --oneline -1
!pip install -q "unsloth==2026.7.3" || pip install -q unsloth
# 裝 unsloth 會換掉 PyTorch 版本，Colab 內建的 torchcodec（影音用，這裡用不到）會因此載入失敗，直接移除
!pip uninstall -y -q torchcodec

In [ ]:
def step(name, cmd, done):
    """跑一個步驟；done 這個檔案已經存在就跳過（之前跑完了）。"""
    if os.path.exists(done):
        print(f"✅ {name}：之前已經完成，跳過")
        return
    print(f"▶ {name} …")
    p = subprocess.run(f"cd {REPO} && {cmd}", shell=True, text=True, stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    lines = p.stdout.splitlines()
    print("\n".join(lines[-25:]))  # 只顯示最後幾行，完整的在檔案裡
    with open(os.path.join(OUT, f"log_{name}.txt"), "w") as f:
        f.write(p.stdout)
    if p.returncode != 0:
        raise RuntimeError(f"{name} 失敗，請把 {OUT}/log_{name}.txt 交給 Claude")
    print(f"✅ {name} 完成")

## 3. 檢查資料（不用 GPU，幾秒鐘）

In [ ]:
step("資料檢查", "python finetune/train.py --dry-run", os.path.join(OUT, "_none_"))

## 4. 沒微調的 4B（基準）
約 30 分鐘。

In [ ]:
step("基準4B", "python finetune/evaluate.py --model unsloth/Qwen3.5-4B",
     os.path.join(OUT, "eval_Qwen3.5-4B_eval.txt"))

## 5. 微調
約 45～60 分鐘。中斷的話再跑一次，會從存檔點接著訓練。

In [ ]:
step("微調", "python finetune/train.py --steps 200",
     os.path.join(OUT, "parser-lora", "train_args.json"))

## 6. 微調後的 4B
約 1 小時：先逐筆比對條件，再讓整個機器人跑 153 個情境、逐句對答案。

In [ ]:
LORA = os.path.join(OUT, "parser-lora")
step("微調後4B", f"python finetune/evaluate.py --model {LORA} --bot",
     os.path.join(OUT, "bot_parser-lora.txt"))

## 7.（時間夠再跑）把 13 條保險規則全部關掉
約 30 分鐘。看微調後的模型不靠規則能做到多好。

In [ ]:
step("規則全關", f"python finetune/evaluate.py --model {LORA} --bot --off homophone,fill_from_rules,near_rule,avoid_rule,pref_off,budget_min_words,switching,budget_from_rules,budget_up_words,category_name,unsure,total_words,choose_on_avoid",
     os.path.join(OUT, "bot_parser-lora_off-homophone+fill_from_rules+near_rule+avoid_rule+pref_off+budget_min_words+switching+budget_from_rules+budget_up_words+category_name+unsure+total_words+choose_on_avoid.txt"))

## 8. 打包結果、下載

In [ ]:
import glob, shutil, zipfile
for f in sorted(glob.glob(os.path.join(OUT, "eval_*.txt"))):
    print(open(f, encoding="utf-8").read().split("\n\n")[0], "\n")
zip_path = "/content/results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(os.path.join(OUT, "*.txt")):
        z.write(f, os.path.basename(f))
    for f in glob.glob(os.path.join(OUT, "parser-lora", "*")):
        z.write(f, os.path.join("parser-lora", os.path.basename(f)))
print(f"results.zip：{os.path.getsize(zip_path) / 1024**2:.0f} MB")
from google.colab import files
files.download(zip_path)